# 02 - Preprocessing, Splitting and Leakage Control

The split is created once here and written to `data/processed/`. Every later
step reads the same files, so the test partition cannot change between
experiments.

In [1]:
import os, sys, pathlib
root = pathlib.Path.cwd()
while not (root / "src" / "config.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
os.chdir(root)          # relative paths in the notebooks resolve from the root
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 160)
print("project root:", root)

project root: /home/shreyas/projects/dry-bean-classification


## 1. Stratified split

In [2]:
from src.data_loader import load_dataset
from src.preprocessing import split_data, persist_split
split = split_data(load_dataset())
persist_split(split)
split.summary()

{'random_state': 42,
 'test_size': 0.2,
 'stratified': True,
 'train_rows': 10888,
 'test_rows': 2723,
 'train_pct': 79.99,
 'test_pct': 20.01,
 'train_class_counts': {'SEKER': 1621,
  'BARBUNYA': 1057,
  'BOMBAY': 418,
  'CALI': 1304,
  'DERMASON': 2837,
  'HOROZ': 1542,
  'SIRA': 2109},
 'test_class_counts': {'SEKER': 406,
  'BARBUNYA': 265,
  'BOMBAY': 104,
  'CALI': 326,
  'DERMASON': 709,
  'HOROZ': 386,
  'SIRA': 527}}

Stratification keeps the class proportions of the full dataset inside
both partitions; without it the smallest class (BOMBAY, 522 beans) could lose
several percent of its already small test representation.

## 2. Why scaling must sit inside the pipeline

In [3]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(split.X_train)          # fitted on TRAIN only
print("train mean Area :", round(scaler.mean_[0], 2))
print("test  mean Area :", round(split.X_test["Area"].mean(), 2))

train mean Area : 53018.07
test  mean Area : 53169.09


In [4]:
from src.config import FEATURES
raw = split.X_train[FEATURES].agg(["min", "max"]).T
raw["range"] = raw["max"] - raw["min"]
raw.round(3).sort_values("range", ascending=False).head(6)

,min,max,range
ConvexArea,20684.000,251082.000,230398.000
Area,20420.000,248424.000,228004.000
Perimeter,524.736,1895.940,1371.204
MajorAxisLength,183.601,726.373,542.772
EquivDiameter,161.244,562.408,401.165
MinorAxisLength,122.513,460.198,337.686


Area spans 20,420-254,616 pixels while `ShapeFactor1` stays below 0.011.
A distance-based model fitted on raw values would be driven almost entirely by
Area. Fitting the scaler on the whole dataset before splitting would leak test
statistics into training, so `src.preprocessing.build_pipeline` places
`StandardScaler` as the first pipeline step and only ever fits it on training
folds.

## 3. Pipeline construction

In [5]:
from src.preprocessing import build_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
print(build_pipeline(LogisticRegression(), scale=True))
print(build_pipeline(RandomForestClassifier(n_estimators=10), scale=False))

Pipeline(steps=[('scaler', StandardScaler()), ('model', LogisticRegression())])
Pipeline(steps=[('model', RandomForestClassifier(n_estimators=10))])


Tree ensembles receive the raw features: a split on `Area < 60,000` is
unchanged by a monotone rescaling, so standardisation would only add cost.

## 4. Duplicate and outlier policy

In [6]:
import pandas as pd
train = pd.read_csv("data/processed/train.csv")
print("duplicate rows kept:", int(train.duplicated().sum()))
print(train.groupby("Class")["Area"].agg(["count", "min", "max"]).round(1))

duplicate rows kept: 39
          count     min     max
Class                          
BARBUNYA   1057   41487  115967
BOMBAY      418  114004  248424
CALI       1304   45504  116272
DERMASON   2837   20420   42147
HOROZ      1542   33006   81929
SEKER      1621   28395   61150
SIRA       2109   31519   63612


Duplicates were retained (pixel quantisation, not data entry errors) and
no outliers were removed, because the extreme values are genuine BOMBAY beans.
Both decisions are tested indirectly by the feature ablation in notebook 03.